In [3]:
import torch
import os
import json
import numpy as np
import pandas as pd
import random
import statistics
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib
from plotting_style import *
from risk_control_utils import (get_label_order, rc_main, get_all_confidences, get_all_accuracies, get_ground_truth_by_type, 
                                get_relative_labels, apply_risk_control, load_all_data, get_losses_and_exits_confidence)

In [4]:
# Define a list of all the models and datasets to plot
models = ["facebook/layerskip-llama3-8B", "facebook/layerskip-llama2-7B", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf"]
tokenizers = ["meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf"]
n_early_exits = [32, 32, 32, 32]
datasets = ['sst2', 'trec', 'financial_phrasebank', 'tweeteval_hate', 'tweeteval_feminist', 
            'tweeteval_atheism', 'unnatural', 'ag_news']
# datasets = ['ag_news', 'mqp', 'mrp', 'wnli'] # less hard datasets
# datasets = ['boolean', 'navigation', 'sports', 'web_of_lies'] # BigBench-Hard datasets

In [5]:
# Define base data folder to read from (and associated params for finding the right data files)
use_calibration = True # True or False
fake_labels = True # True or False
confidence_type = 'argmax' # argmax or top2_diff or entropy
precomputed_risk_path = './rc-precomputed/' + ('fake_labels' if fake_labels else '') + ('/calibrated/' if use_calibration else '/uncalibrated/')
precomputed_risk_path += confidence_type + '/'
results_folder = './results' + ('_fake_labels' if fake_labels else '') + ('/calibrated' if use_calibration else '/uncalibrated')
n_demos=60 # 60
with open('fake_labels.json') as f:
    fake_label_map = json.load(f)

In [6]:
# Define plotting params
debug_mode = False
c_color, i_color, z_color = 'tab:blue', 'tab:orange', 'tab:green'
model_colors = {"facebook/layerskip-llama3-8B": 'tab:brown', "facebook/layerskip-llama2-7B": 'tab:pink', "meta-llama/Meta-Llama-3-8B": 'tab:purple', 
               "meta-llama/Llama-2-7B-hf": 'tab:cyan'}
first_exit=15 # for risk control, this sets the earliest layer at which we are allowed to exit
plot_directory = './icl_plots' + ('_fake_labels' if fake_labels else '') + '/' 
plot_directory += ('calibrated' if use_calibration else 'uncalibrated') + '/n_demos_' + str(n_demos) + '/'

In [7]:
# Define lambdas and epsilons
# NOTE: If running with loss_01_conversion=max_0, cannot have epsilon < 0!
stepsize = 0.01
eps_grid = np.arange(0.0, 0.5 + stepsize, stepsize)
lambdas = np.arange(0.0, 1.0 + stepsize, stepsize)[::-1]

In [8]:
# Define loss and risk-control parameters
relative_labels = 'zeroshot_full_model' # zeroshot_full_model or full_model; the predictions over which to compute a relative loss
ground_truth_type = 'true_label' # the ground-truth for computing loss; true_label or zeroshot_full_model
rcp_type = 'ltt'
delta=0.1

# when the risk cannot be controlled for any lambda this is the default loss and exit
# define this as the relative loss
default_loss_uncontrolled_risk = 0 
default_eff_gain_uncontrolled_risk = 0

In [9]:
if debug_mode:
    # Select just a subset of the datasets and models
    datasets = ['financial_phrasebank']
    n_trials=2
else:
    # Prevent displaying figures
    matplotlib.use('Agg')
    n_trials=50

In [10]:
# Pre-compute and save the risk control results matrices
if not os.path.exists(precomputed_risk_path):
    for dataset in datasets:
        for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            # First check that there exists all types of experiments
            if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                    and os.path.exists(base_dir + 'zeroshot.json')):
                # We have all the data
                data = {}
                for expt_type in ['correct', 'incorrect', 'zeroshot']:
                    with open(base_dir + expt_type + '.json', 'r') as file:
                        data[expt_type] = json.load(file)
                    
                c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
                c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
    
                for gt, rel, label in zip([c_gt, i_gt], [c_rel, i_rel], ['correct', 'incorrect']):
                    conf = get_all_confidences(data[label], n_early_exit, label_order, confidence_type, first_exit)
                    acc = get_all_accuracies(data[label], gt, n_early_exit, first_exit)
                    n_cal = int(len(data[label]['0'])/2)
                    # Run the max-0 method
                    losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid, rcp_type, 
                                                                                delta, n_cal, n_trials, default_loss_uncontrolled_risk, 
                                                                                default_eff_gain_uncontrolled_risk, 'max_0')
                    # Save results
                    max0_path = precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/' + label + '/'
                    if not os.path.exists(max0_path):
                        os.makedirs(max0_path)
                    np.save(max0_path + 'losses.npy', losses)
                    np.save(max0_path + 'test_risk.npy', test_risk)
                    np.save(max0_path + 'eff_gains.npy', eff_gains)
                    np.save(max0_path + 'rcp_lams.npy', rcp_lams)
                    # Run the scaling method
                    losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid, rcp_type, 
                                                                                delta, n_cal, n_trials, default_loss_uncontrolled_risk, 
                                                                                default_eff_gain_uncontrolled_risk, 'scaling')
                    # Save results
                    scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/' + label + '/'
                    if not os.path.exists(scaling_path):
                        os.makedirs(scaling_path)
                    np.save(scaling_path + 'losses.npy', losses)
                    np.save(scaling_path + 'test_risk.npy', test_risk)
                    np.save(scaling_path + 'eff_gains.npy', eff_gains)
                    np.save(scaling_path + 'rcp_lams.npy', rcp_lams)
            else:
                print('Missing data: ', dataset, model_name)
        print('Finished', dataset)

In [11]:
# Pre-compute and save the risk control results matrices for combined correct + incorrect demos
precomputed_mixed_path = precomputed_risk_path + 'mixed_correct_incorrect/'
if not os.path.exists(precomputed_mixed_path):
    for dataset in datasets:
        for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            # First check that there exists all types of experiments
            if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                    and os.path.exists(base_dir + 'zeroshot.json')):
                # We have all the data
                data = {}
                for expt_type in ['correct', 'incorrect', 'zeroshot']:
                    with open(base_dir + expt_type + '.json', 'r') as file:
                        data[expt_type] = json.load(file)
                    
                c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
                c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

                # Combine correct and incorrect
                gt, rel = c_gt + i_gt, c_rel + i_rel
                combined_data = {}
                for col in data['correct']:
                    combined_data[col] = data['correct'][col] + data['incorrect'][col]

                conf = get_all_confidences(combined_data, n_early_exit, label_order, confidence_type, first_exit)
                acc = get_all_accuracies(combined_data, gt, n_early_exit, first_exit)
                n_cal = int(len(combined_data['0'])/2)
                
                # Run the max-0 method
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid, rcp_type, 
                                                                            delta, n_cal, n_trials, default_loss_uncontrolled_risk, 
                                                                            default_eff_gain_uncontrolled_risk, 'max_0')
                # Save results
                max0_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
                if not os.path.exists(max0_path):
                    os.makedirs(max0_path)
                np.save(max0_path + 'losses.npy', losses)
                np.save(max0_path + 'test_risk.npy', test_risk)
                np.save(max0_path + 'eff_gains.npy', eff_gains)
                np.save(max0_path + 'rcp_lams.npy', rcp_lams)
                # Run the scaling method
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid, rcp_type, 
                                                                            delta, n_cal, n_trials, default_loss_uncontrolled_risk, 
                                                                            default_eff_gain_uncontrolled_risk, 'scaling')
                # Save results
                scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
                if not os.path.exists(scaling_path):
                    os.makedirs(scaling_path)
                np.save(scaling_path + 'losses.npy', losses)
                np.save(scaling_path + 'test_risk.npy', test_risk)
                np.save(scaling_path + 'eff_gains.npy', eff_gains)
                np.save(scaling_path + 'rcp_lams.npy', rcp_lams)
            else:
                print('Missing data: ', dataset, model_name)
        print('Finished', dataset)

# Combined Correct + Incorrect

In [13]:
def compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits):
    test_risk_e, test_risk_err, eff_gains_e, eff_gains_err = [], [], [], []
    for e, eps in enumerate(eps_grid):
        lam_id = rcp_lams[e]
        if lam_id is None:
            test_risk_e.append(default_loss_uncontrolled_risk)
            eff_gains_e.append(default_eff_gain_uncontrolled_risk)
            test_risk_err.append(0)
            eff_gains_err.append(0)
        else:
            test_risk_e.append(losses[lam_id].mean())
            eff_gains_e.append(exits[lam_id].mean())
            test_risk_err.append(losses[lam_id].std(axis=0) / np.sqrt(losses[lam_id].shape[0]))
            eff_gains_err.append(exits[lam_id].std(axis=0) / np.sqrt(exits[lam_id].shape[0]))

    return np.array(test_risk_e), np.array(eff_gains_e), np.array(test_risk_err), np.array(eff_gains_err)

In [113]:
def plot_risk_control(ax, eps_grid, losses, test_risk, label, color, linestyle):
    risk_mean, risk_err = test_risk.mean(axis=0), test_risk.std(axis=0) / np.sqrt(test_risk.shape[0])
    ax.plot(eps_grid, risk_mean, label=label, color=color, linestyle=linestyle)
    ax.fill_between(eps_grid, risk_mean - risk_err, risk_mean + risk_err, alpha=0.2, color=color)
    # add a diagonal line and axis labels
    ax.plot([min(eps_grid), max(eps_grid)], [min(eps_grid), max(eps_grid)], 'k--')
    ax.set_ylabel('Risk')
    ax.set_xlabel('epsilon')

In [15]:
def plot_separate_risk(ax, eps_grid, c_mean, c_err, i_mean, i_err, color):
    ax.plot(eps_grid, c_mean, label='correct', color=color, linestyle='solid')
    ax.fill_between(eps_grid, c_mean - c_err, c_mean + c_err, alpha=0.2, color=color)
    ax.plot(eps_grid, i_mean, label='incorrect', color=color, linestyle='dotted')
    ax.fill_between(eps_grid, i_mean - i_err, i_mean + i_err, alpha=0.2, color=color)
    ax.set_ylabel('Risk')
    ax.set_xlabel('epsilon')

In [123]:
# Safety of incorrect demos
for dataset in datasets:
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        # First check that all experiment results are precomputed
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            # load the data
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels('full_model', data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            
            # get the pre-computed lambda-hat's
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
            rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)

            # Plot incorrect full-model risk relative to incorrect full-model
            conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, first_exit)
            acc = get_all_accuracies(data['incorrect'], i_gt, n_early_exit, first_exit)
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, i_rel, i_gt)
            i_mean, _, i_err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)

            # plot a single line for the relative risk of incorrect demos vs the full model
            ax.plot(eps_grid, i_mean, label=label, color=model_colors[model_name], linestyle='solid')
            ax.fill_between(eps_grid, i_mean - i_err, i_mean + i_err, alpha=0.2, color=model_colors[model_name])
        else:
            print('Missing data:', model_name, dataset)

    ax.set_ylabel('Loss Relative to Full-Model')
    ax.set_xlabel('epsilon')
    ax.legend()
    plt.tight_layout()

    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'mixed_correct_incorrect/incorrect_demos_safety/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


In [135]:
# Split plots
for dataset in datasets:
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        # First check that all experiment results are precomputed
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            # load the data
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]
            
            # get the pre-computed lambda-hat's
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
            rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)

            # compute + plot epsilon vs risk for correct demos
            conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, first_exit)
            acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, first_exit)
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, c_rel, c_gt)
            c_mean, _, c_err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)

            # compute + plot epsilon vs risk for incorrect demos
            conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, first_exit)
            acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, first_exit)
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, c_rel, c_gt)
            i_mean, _, i_err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)

            # plot correct/incorrect separately
            plot_separate_risk(ax, eps_grid, c_mean, c_err, i_mean, i_err, model_colors[model_name])
        else:
            print('Missing data:', model_name, dataset)

    lines2 = [Line2D([0], [0], color='black', lw=1, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
    labels2 = ['incorrect', 'correct']
    
    lines1 = [Line2D([0], [0], color=model_colors[model], lw=1, linestyle='-',) for model in models]
    lines1 += lines2
    labels1 = [x.split('/')[1] for x in models] + labels2
    
    legend1 = plt.legend(lines1, labels1, loc='upper left',)
    ax.plot([min(eps_grid), max(eps_grid)], [min(eps_grid), max(eps_grid)], 'k--')
    plt.tight_layout()

    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'mixed_correct_incorrect/rc_split/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


In [137]:
# Risk control plots
for dataset in datasets:
    fig, ax = plt.subplots(1,2,figsize=(10,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        # First check that all experiment results are precomputed
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            # load the data
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]
            
            # get the pre-computed lambda-hat's
            scaling_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
            rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)

            # plot epsilon vs risk for ALL demos (50-50 split)
            test_risk, losses = np.load(scaling_path + 'test_risk.npy'), np.load(scaling_path + 'losses.npy')
            plot_risk_control(ax[0], eps_grid, losses, test_risk, model_name, model_colors[model_name], 'solid')

            # compute + plot epsilon vs risk for correct demos
            conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, first_exit)
            acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, first_exit)
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, c_rel, c_gt)
            c_mean, _, c_err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)

            # compute + plot epsilon vs risk for incorrect demos
            conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, first_exit)
            acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, first_exit)
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, c_rel, c_gt)
            i_mean, _, i_err, _ = compute_rc_fixed_lambda(eps_grid, rcp_type, rcp_lams, losses, exits)

            # plot correct/incorrect separately
            plot_separate_risk(ax[1], eps_grid, c_mean, c_err, i_mean, i_err, model_colors[model_name])
        else:
            print('Missing data:', model_name, dataset)

    lines2 = [Line2D([0], [0], color='black', lw=1, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
    labels2 = ['incorrect', 'correct']
    
    lines1 = [Line2D([0], [0], color=model_colors[model], lw=1, linestyle='-',) for model in models]
    labels1 = [x.split('/')[1] for x in models]
    
    legend1 = ax[0].legend(lines1, labels1, loc='upper left',)
    legend2 = ax[1].legend(lines2, labels2, loc='upper left',)
    plt.tight_layout()

    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'mixed_correct_incorrect_rc/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


In [99]:
# Risk control plots - with only risk control on combined data
for dataset in datasets:
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        # First check that all experiment results are precomputed
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            label_order = get_label_order(dataset, tokenizer)
            if fake_labels:
                label_order = [fake_label_map[x] for x in label_order]
            
            # load the data
            base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Combine correct and incorrect
            gt, rel = c_gt + i_gt, c_rel + i_rel
            combined_data = {}
            for col in data['correct']:
                combined_data[col] = data['correct'][col] + data['incorrect'][col]
            
            # get the pre-computed lambda-hat's
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
            rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)

            # plot epsilon vs risk for ALL demos (50-50 split)
            test_risk, losses = np.load(scaling_path + 'test_risk.npy'), np.load(scaling_path + 'losses.npy')
            plot_risk_control(ax, eps_grid, losses, test_risk, model_name.split('/')[1], model_colors[model_name], 'solid')
        else:
            print('Missing data:', model_name, dataset)

    ax.legend()
    plt.tight_layout()

    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'mixed_correct_incorrect_rc/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


In [91]:
# Plot risk control results and exit layer, and with comparison to the other approach
# Also study the average efficiency gains
for dataset in datasets:
    fig, ax = plt.subplots(1,2,figsize=(10,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
            rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            
            # max-0
            max0_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
            losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, label + ' clipped', model_colors[model_name], 'dashed')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, label + ' clipped', model_colors[model_name], 'dashed')
            # Scaling
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/' 
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, label + ' with transformation', model_colors[model_name], 'solid')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, label + ' with transformation', model_colors[model_name], 'solid')
        else:
            print('Missing data: ', dataset, model_name)

    lines2 = [Line2D([0], [0], color='black', lw=1, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
    labels2 = ['clipped risk', 'risk transformation']
    
    lines1 = [Line2D([0], [0], color=model_colors[model], lw=1, linestyle='-',) for model in models]
    labels1 = [x.split('/')[1] for x in models]
    
    legend1 = ax[0].legend(lines1, labels1, loc='upper left',)
    legend2 = ax[1].legend(lines2, labels2, loc='upper right',)
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'mixed_correct_incorrect/risk_control_with_comparison/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


In [ ]:
# Plot JUST efficiency gains
for dataset in datasets:
    fig, ax = plt.subplots(1,2,figsize=(10,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'):
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/'
            rcp_lams = np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            
            # max-0
            max0_path = precomputed_mixed_path + 'max0/' + dataset + '/' + model_name + '/'
            losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, label + ' clipped', model_colors[model_name], 'dashed')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, label + ' clipped', model_colors[model_name], 'dashed')
            # Scaling
            scaling_path = precomputed_mixed_path + 'scaling/' + dataset + '/' + model_name + '/' 
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, label + ' with transformation', model_colors[model_name], 'solid')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, label + ' with transformation', model_colors[model_name], 'solid')
        else:
            print('Missing data: ', dataset, model_name)

    lines2 = [Line2D([0], [0], color='black', lw=1, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
    labels2 = ['clipped risk', 'risk transformation']
    
    lines1 = [Line2D([0], [0], color=model_colors[model], lw=1, linestyle='-',) for model in models]
    labels1 = [x.split('/')[1] for x in models]
    
    legend1 = ax[0].legend(lines1, labels1, loc='upper left',)
    legend2 = ax[1].legend(lines2, labels2, loc='upper right',)
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'mixed_correct_incorrect/risk_control_with_comparison/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()

    print('Finished', dataset)

# Lambda vs Accuracy

In [18]:
def plot_lambda_vs_risk(ax, lambdas, losses, label, color, linestyle):
    # Create plot of lambda vs risk
    ax.plot(lambdas, losses.mean(axis=1), label=label, color=color, linestyle=linestyle)
    # Add error bars
    err = np.std(losses, axis=1) / np.sqrt(losses.shape[1])
    ax.fill_between(lambdas, losses.mean(axis=1) - err, losses.mean(axis=1) + err, alpha=0.2, color=color)
    ax.set_xlabel('Lambda')
    ax.set_ylabel('Empirical Risk')

In [73]:
# Plot lambda vs relative accuracy
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        label_order = get_label_order(dataset, tokenizer)
        if fake_labels:
            label_order = [fake_label_map[x] for x in label_order]
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)

            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)

            # Plot lambda vs correct accuracy
            conf = get_all_confidences(data['correct'], n_early_exit, label_order, confidence_type, first_exit)
            acc = get_all_accuracies(data['correct'], c_gt, n_early_exit, first_exit)
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, c_gt)
            ax.plot(lambdas, 1-losses.mean(axis=1), label='correct demos', color=c_color)

            # Plot lambda vs incorrect accuracy
            conf = get_all_confidences(data['incorrect'], n_early_exit, label_order, confidence_type, first_exit)
            acc = get_all_accuracies(data['incorrect'], c_gt, n_early_exit, first_exit)
            losses, exits = get_losses_and_exits_confidence(conf, acc, lambdas, None, i_gt)
            ax.plot(lambdas, 1-losses.mean(axis=1), label='incorrect demos', color=i_color)

            # Plot full-model performance as horizontal lines
            correct_acc = [1 if p == t else 0 for p,t in zip(data['correct'][str(n_early_exit-1)], c_gt)]
            zeroshot_acc = [1 if p == t else 0 for p,t in zip(data['zeroshot'][str(n_early_exit-1)], z_gt)]
            #correct_full_model = [c-z for c,z in zip(correct_acc, zeroshot_acc)]
            correct_full_model = correct_acc
            ax.axhline(y=sum(correct_full_model) / len(correct_full_model), color=c_color, linestyle='--', linewidth=2, label='correct full model')
            #ax.axhline(y=sum(zeroshot_acc) / len(zeroshot_acc), color=z_color, linestyle='--', linewidth=2, label='zeroshot full model')
            plt.axvline(x=0.85, linestyle='dotted', color='red')
            
            ax.set_xlabel('Lambda')
            ax.set_ylabel('Calibrated Accuracy (Relative to Zero-Shot)')
            ax.legend()
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'lambda_vs_accuracy/confidence_' + confidence_type + '/' 
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.png')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)
    
    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Risk Control

In [21]:
def plot_efficiency_gains(ax, eps_grid, eff_gains, label, color, linestyle):
    exit_mean, exit_err = n_early_exit - eff_gains.mean(axis=0), eff_gains.std(axis=0) / np.sqrt(eff_gains.shape[0])
    ax.plot(eps_grid, exit_mean, label=label, color=color, linestyle=linestyle)
    ax.fill_between(eps_grid, exit_mean - exit_err, exit_mean + exit_err, alpha=0.2, color=color)
    ax.set_ylabel('Average Exit Layer')
    ax.set_xlabel('epsilon')

In [22]:
# Plot risk control results and exit layer
# ONLY using the scaling approach, and not including zero-shot! One plot per dataset. 
for dataset in datasets:
    fig, ax = plt.subplots(1,2,figsize=(10,5))
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        # First check that all experiment results are precomputed
        if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct'):
            # Run the scaling method ONLY - correct demos
            scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/'
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, 'correct', model_colors[model_name], 'solid')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, 'correct', model_colors[model_name], 'solid')
        
            # Run the scaling method ONLY - incorrect demos
            scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/incorrect/'
            losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
            eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
            plot_risk_control(ax[0], eps_grid, losses, test_risk, 'incorrect', model_colors[model_name], 'dashed')
            plot_efficiency_gains(ax[1], eps_grid, eff_gains, 'incorrect', model_colors[model_name], 'dashed')
        else:
            print('Missing data: ', dataset, model_name)

    lines2 = [Line2D([0], [0], color='black', lw=1, linestyle=sty, alpha=0.5) for sty in ['dashed', 'solid']]
    labels2 = ['incorrect', 'correct']
    
    lines1 = [Line2D([0], [0], color=model_colors[model], lw=1, linestyle='-',) for model in models]
    labels1 = [x.split('/')[1] for x in models]
    
    legend1 = ax[0].legend(lines1, labels1, loc='upper right',)
    legend2 = ax[1].legend(lines2, labels2, loc='upper right',)
    plt.tight_layout()
    
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'risk_control/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()
    
    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


## Risk Control w Comparison

In [24]:
# Plot risk control results and exit layer, and with comparison to the other approach
# Also study the average efficiency gains
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/'):
            fig, ax = plt.subplots(1,2,figsize=(10,5))            
            for label, color in zip(['correct', 'incorrect'], [c_color, i_color]):
                # max-0
                max0_path = precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/' + label + '/'
                losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
                eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
                plot_risk_control(ax[0], eps_grid, losses, test_risk, label + ' clipped', color, 'dashed')
                plot_efficiency_gains(ax[1], eps_grid, eff_gains, label + ' clipped', color, 'dashed')
                # Scaling
                scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/' + label + '/'
                losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
                eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
                plot_risk_control(ax[0], eps_grid, losses, test_risk, label + ' with transformation', color, 'solid')
                plot_efficiency_gains(ax[1], eps_grid, eff_gains, label + ' with transformation', color, 'solid')
            ax[0].legend()
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'risk_control_with_comparison/confidence_' + confidence_type + '/' 
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.png')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)

    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Efficiency Gains

In [26]:
# Evaluate efficiency gain difference between scaling vs max0
eps_to_evaluate = 0.05
eps_index = np.where(eps_grid == eps_to_evaluate)[0][0]
avg_incorrect_diff, avg_correct_diff, total_n = 0, 0, 0
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/'):
            total_n += 1
            # Correct demos
            eff_gains = np.load(precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/correct/' + 'eff_gains.npy')
            max0_gains = eff_gains.mean(axis=0)[eps_index]
            eff_gains = np.load(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/' + 'eff_gains.npy')
            scaling_gains = eff_gains.mean(axis=0)[eps_index]
            avg_correct_diff += (scaling_gains - max0_gains)
            # Incorrect demos
            eff_gains = np.load(precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/incorrect/' + 'eff_gains.npy')
            max0_gains = eff_gains.mean(axis=0)[eps_index]
            eff_gains = np.load(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/incorrect/' + 'eff_gains.npy')
            scaling_gains = eff_gains.mean(axis=0)[eps_index]
            avg_incorrect_diff += (scaling_gains - max0_gains)
        else:
            print('Missing data: ', dataset, model_name)

avg_correct_diff, avg_incorrect_diff = avg_correct_diff / total_n, avg_incorrect_diff / total_n
print('Avg diff - correct examples:', avg_correct_diff, 'Percent:', avg_correct_diff/17*100)
print('Avg diff - incorrect examples:', avg_incorrect_diff, 'Percent:', avg_incorrect_diff/17*100)

Avg diff - correct examples: 9.246036574615932 Percent: 54.388450438917246
Avg diff - incorrect examples: 1.1456249654966695 Percent: 6.738970385274526


In [141]:
# Plot risk control results and exit layer, and with comparison to the other approach
# Also study the average efficiency gains
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        if os.path.exists(precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/correct/'):
            fig, ax = plt.subplots(1,1,figsize=(5,5))            
            for label, color in zip(['correct', 'incorrect'], [c_color, i_color]):
                # max-0
                max0_path = precomputed_risk_path + 'max0/' + dataset + '/' + model_name + '/' + label + '/'
                losses, test_risk = np.load(max0_path + 'losses.npy'), np.load(max0_path + 'test_risk.npy')
                eff_gains, rcp_lams = np.load(max0_path + 'eff_gains.npy'), np.load(max0_path + 'rcp_lams.npy', allow_pickle=True)
                plot_efficiency_gains(ax, eps_grid, eff_gains, label + ' clipped', color, 'dashed')
                # Scaling
                scaling_path = precomputed_risk_path + 'scaling/' + dataset + '/' + model_name + '/' + label + '/'
                losses, test_risk = np.load(scaling_path + 'losses.npy'), np.load(scaling_path + 'test_risk.npy')
                eff_gains, rcp_lams = np.load(scaling_path + 'eff_gains.npy'), np.load(scaling_path + 'rcp_lams.npy', allow_pickle=True)
                plot_efficiency_gains(ax, eps_grid, eff_gains, label + ' with transformation', color, 'solid')
            ax.legend()
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'eff_gains_with_comparison/confidence_' + confidence_type + '/' 
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.png')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)

    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


# Accuracy vs Layer

In [28]:
# Accuracy vs Layer plots (for Overthinking section)
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        fig, ax = plt.subplots(1,1,figsize=(5,5))
        label_order = get_label_order(dataset, tokenizer)
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            data = {}
            for expt_type in ['correct', 'incorrect', 'zeroshot']:
                with open(base_dir + expt_type + '.json', 'r') as file:
                    data[expt_type] = json.load(file)
            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, data['correct'], data['incorrect'], data['zeroshot'], n_early_exit)
            
            for gt, label, color in zip([c_gt, i_gt, z_gt], ['correct', 'incorrect', 'zeroshot'], [c_color, i_color, z_color]):
                acc = get_all_accuracies(data[label], gt, n_early_exit, 0)
                ax.plot([i for i in range(n_early_exit)], acc.mean(axis=0), label=label, color=color)

            ax.legend()
            ax.set_xlabel('Layer')
            ax.set_ylabel('Calibrated Accuracy')
            plt.tight_layout()
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'loss_vs_layer/' 
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + dataset + '_' + model_name.split('/')[1] + '.png')
        
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)
    
    print('Finished', dataset)

Finished sst2
Finished trec
Finished financial_phrasebank
Finished tweeteval_hate
Finished tweeteval_feminist
Finished tweeteval_atheism
Finished unnatural
Finished ag_news


In [29]:
# Plot confidence of each layer's prediction through layers
# for dataset in datasets:
#     fig, ax = plt.subplots(1,len(models),figsize=(5*len(models),5))
#     plt.suptitle(dataset)
    
#     for model_idx in range(len(models)):
#         model_name, n_early_exit, tokenizer = models[model_idx], n_early_exits[model_idx], tokenizers[model_idx]
#         layers = np.arange(n_early_exit)
#         label_order = get_label_order(dataset, tokenizer)
        
#         data_path = results_folder + dataset + '/' + str(n_demos) + '/' + model_name + '/'
#         if os.path.exists(data_path + 'zeroshot.csv'):
#             # Load data
#             correct = pd.read_csv(data_path + 'correct.csv', engine='python', on_bad_lines='warn').dropna()
#             incorrect = pd.read_csv(data_path + 'incorrect.csv', engine='python', on_bad_lines='warn').dropna()
#             zeroshot = pd.read_csv(data_path + 'zeroshot.csv', engine='python', on_bad_lines='warn').dropna()

#             c_conf = get_all_confidences(correct, n_early_exit, label_order, confidence_type)
#             i_conf = get_all_confidences(incorrect, n_early_exit, label_order, confidence_type)
#             z_conf = get_all_confidences(zeroshot, n_early_exit, label_order, confidence_type)

#             ax[model_idx].plot(layers, c_conf.mean(axis=0), label='correct')
#             ax[model_idx].plot(layers, i_conf.mean(axis=0), label='incorrect')
#             ax[model_idx].plot(layers, z_conf.mean(axis=0), label='zeroshot')
#             ax[model_idx].set_xlabel('Layer')
#             ax[model_idx].set_ylabel('Confidence')
#             ax[model_idx].set_title(model_name)

#     if debug_mode:
#         # Display the image
#         plt.show()
#     else:
#         # Save out the image
#         path = plot_directory + 'confidences/' + confidence_type + '/'
#         if not os.path.exists(path):
#             os.makedirs(path)
#         plt.savefig(path + dataset + '.png')

#     # Close plots to save memory
#     matplotlib.pyplot.close()
#     print('Finished ', dataset)